# DeepFALCON GSoC 2026 – Common Task 2
## Graph Neural Network for Quark/Gluon Jet Classification

**Pipeline:**
```
Image (3×125×125)
  └─ Step 1: Non-zero pixels → Point Cloud  (η, φ, E_ECAL, E_HCAL, E_Track, E_tot, ΔR)
       └─ Step 2: k-NN graph in (η,φ,E_tot) space
            └─ Step 3: DGCNN/EdgeConv GNN → quark vs gluon
```

**Model: JetGNN (DGCNN-style with EdgeConv)**
- 3 × EdgeConv blocks (7→64→128→256)
- Dual pooling: GlobalMean + GlobalMax → 512-d
- FC head → 2 classes

In [10]:
name = "Sakina "
print(f"Hello me {name} bol ri hun!")

Hello me Sakina  bol ri hun!


In [22]:
# Install dependencies (run once)
!pip install torch torchvision
!pip install torch-geometric torch-cluster torch-scatter
!pip install h5py matplotlib scikit-learn networkx tqdm


  error: subprocess-exited-with-error
  
  python setup.py bdist_wheel did not run successfully.
  exit code: 1
  
  [29 lines of output]
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\fps.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\graclus.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\grid.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\knn.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\nearest.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\radius.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\rw.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\sampler.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying torch_cluster\testing.py -> build\lib.win-amd64-cpython-312\torch_cluster
  copying


  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached torch_scatter-2.1.2.tar.gz (108 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py clean for torch-cluster
  Running setup.py clean for torch-scatter
Failed to build torch-cluster torch-scatter


In [23]:
import os, random, time, math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import h5py, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.data import Data, Batch
from torch_geometric.nn import EdgeConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected
from torch_cluster import knn_graph
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Config ──────────────────────────────────────────────────
DATA_PATH   = 'quark-gluon_data-set_n139306.hdf5'   # <- file path to the dataset that cna be updated
K_NEIGHBORS = 8
MAX_NODES   = 400
BATCH_SIZE  = 32
N_EPOCHS    = 5
LR          = 1e-3
DROPOUT     = 0.3
MAX_SAMPLES = 5000   # can be changed for quick test
OUT_DIR     = 'outputs_gnn'
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE  = 125
ETA_RANGE = 2.6
PHI_RANGE = 2 * math.pi

ModuleNotFoundError: No module named 'torch_cluster'

## Step 1 – Image → Point Cloud

In [ ]:
def image_to_pointcloud(img, threshold=0.0):
    """
    img: (3,125,125)
    Returns features (N,7): [η, φ, E_ECAL, E_HCAL, E_Tracks, E_tot, ΔR]
    """
    C, H, W = img.shape
    eta_vals = (np.arange(H)/(H-1))*ETA_RANGE - ETA_RANGE/2
    phi_vals = (np.arange(W)/(W-1))*PHI_RANGE - PHI_RANGE/2
    mask = np.any(img > threshold, axis=0)   # active pixels
    ri, ci = np.where(mask)
    if len(ri) == 0: return None

    eta = eta_vals[ri]; phi = phi_vals[ci]
    e_ecal  = img[0, ri, ci]
    e_hcal  = img[1, ri, ci]
    e_track = img[2, ri, ci]
    e_tot   = e_ecal + e_hcal + e_track

    # energy-weighted centroid → ΔR per node
    w = e_tot.sum()
    eta_c = np.sum(eta*e_tot)/w if w>0 else 0
    phi_c = np.sum(phi*e_tot)/w if w>0 else 0
    delta_r = np.sqrt((eta-eta_c)**2 + (phi-phi_c)**2)

    return np.stack([eta,phi,e_ecal,e_hcal,e_track,e_tot,delta_r],
                    axis=1).astype(np.float32)  # (N,7)


def pointcloud_to_graph(features, k=8, max_nodes=400):
    """k-NN graph in (η,φ,E_tot) space. Returns PyG Data."""
    if features.shape[0] > max_nodes:
        idx = np.random.choice(features.shape[0], max_nodes, replace=False)
        features = features[idx]
    x   = torch.tensor(features, dtype=torch.float32)
    pos = x[:, [0,1,5]]                              # (η,φ,E_tot)
    edge_index = to_undirected(knn_graph(pos, k=k, loop=False))
    src, dst   = edge_index
    d_eta  = x[dst,0]-x[src,0]; d_phi = x[dst,1]-x[src,1]
    d_etot = x[dst,5]-x[src,5]; d_r   = torch.sqrt(d_eta**2+d_phi**2)
    edge_attr = torch.stack([d_eta,d_phi,d_etot,d_r], dim=1)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)

print('Point cloud functions defined.')

## Dataset

In [ ]:
class JetGraphDataset(Dataset):
    def __init__(self, filepath, k=8, max_nodes=400, max_samples=None):
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            X = f['X' if 'X' in keys else 'jetImage'][:]
            Y = f['y' if 'y' in keys else 'jetLabel'][:]
        if X.ndim==4 and X.shape[-1]==3: X=X.transpose(0,3,1,2)
        X = X.astype(np.float32); X = np.log1p(X)
        for c in range(3):
            p=np.percentile(X[:,c],99)
            if p>0: X[:,c]=np.clip(X[:,c]/p,0,1)
        if max_samples: X=X[:max_samples]; Y=Y[:max_samples]
        self.X=X; self.Y=Y.astype(np.int64); self.k=k; self.max_nodes=max_nodes
        print(f'Loaded {len(X)} events')

    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        feats = image_to_pointcloud(self.X[idx])
        if feats is None: feats = np.zeros((1,7),dtype=np.float32)
        g = pointcloud_to_graph(feats, self.k, self.max_nodes)
        g.y = torch.tensor([int(self.Y[idx])], dtype=torch.long)
        return g

def collate_fn(batch): return Batch.from_data_list(batch)

full_ds = JetGraphDataset(DATA_PATH, k=K_NEIGHBORS,
                           max_nodes=MAX_NODES, max_samples=MAX_SAMPLES)
n_tr=int(.8*len(full_ds)); n_vl=int(.1*len(full_ds)); n_te=len(full_ds)-n_tr-n_vl
train_ds,val_ds,test_ds = random_split(full_ds,[n_tr,n_vl,n_te],
    generator=torch.Generator().manual_seed(SEED))
train_loader=DataLoader(train_ds,BATCH_SIZE,shuffle=True, collate_fn=collate_fn,num_workers=2)
val_loader  =DataLoader(val_ds,  BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2)
test_loader =DataLoader(test_ds, BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2)
print(f'Train/Val/Test: {n_tr}/{n_vl}/{n_te}')

## Visualise a Jet as a Graph

In [ ]:
def plot_jet_graph(graph, title='Jet'):
    pos_np = graph.x[:,[0,1]].numpy()
    e_tot  = graph.x[:,5].numpy()
    ei     = graph.edge_index.numpy()
    fig,axes = plt.subplots(1,2,figsize=(12,5),facecolor='#0d0d0d')

    ax=axes[0]; ax.set_facecolor('#1a1a2e')
    sc=ax.scatter(pos_np[:,0],pos_np[:,1],c=e_tot,cmap='inferno',s=10,alpha=0.9)
    plt.colorbar(sc,ax=ax).set_label('E_tot',color='white')
    ax.set_xlabel('η',color='#aaa'); ax.set_ylabel('φ',color='#aaa')
    ax.set_title(f'Point Cloud\n{len(pos_np)} nodes',color='white')
    ax.tick_params(colors='white')

    ax2=axes[1]; ax2.set_facecolor('#1a1a2e')
    for s,d in zip(ei[0],ei[1]):
        ax2.plot([pos_np[s,0],pos_np[d,0]],[pos_np[s,1],pos_np[d,1]],
                 color='#334466',lw=0.3,alpha=0.5)
    ax2.scatter(pos_np[:,0],pos_np[:,1],c=e_tot,cmap='inferno',s=8,alpha=0.9,zorder=5)
    ax2.set_xlabel('η',color='#aaa'); ax2.set_ylabel('φ',color='#aaa')
    ax2.set_title(f'k-NN Graph (k={K_NEIGHBORS})\n{ei.shape[1]} edges',color='white')
    ax2.tick_params(colors='white')

    fig.suptitle(title,color='white',fontsize=12)
    plt.tight_layout(); plt.show()

label_map={0:'Gluon',1:'Quark'}
for idx in range(min(50, len(full_ds))):
    g = full_ds[idx]
    if g.y.item()==1:      # find a quark
        plot_jet_graph(g, title=f'Quark Jet (idx={idx})')
        break
for idx in range(min(50, len(full_ds))):
    g = full_ds[idx]
    if g.y.item()==0:      # find a gluon
        plot_jet_graph(g, title=f'Gluon Jet (idx={idx})')
        break

## Point Cloud Statistics: Quark vs Gluon

In [ ]:
n_nodes_q, n_nodes_g, dr_q, dr_g = [], [], [], []
for i in range(min(2000,len(full_ds))):
    g = full_ds[i]; lbl=g.y.item()
    nn_ = g.x.shape[0]; dr_ = g.x[:,6].mean().item()
    if lbl==1: n_nodes_q.append(nn_); dr_q.append(dr_)
    else:      n_nodes_g.append(nn_); dr_g.append(dr_)

fig,axes=plt.subplots(1,2,figsize=(11,4),facecolor='#0d0d0d')
CMAPS={'Gluon':'#ff6e40','Quark':'#00e5ff'}
for ax,(title,dq,dg,xl) in zip(axes,[
    ('Nodes per event',n_nodes_q,n_nodes_g,'# nodes'),
    ('Mean ΔR per event',dr_q,dr_g,'ΔR')
]):
    ax.set_facecolor('#1a1a2e'); ax.tick_params(colors='white')
    bins=40
    ax.hist(dg,bins=bins,density=True,alpha=0.6,color=CMAPS['Gluon'],
            label='Gluon',histtype='stepfilled')
    ax.hist(dq,bins=bins,density=True,alpha=0.6,color=CMAPS['Quark'],
            label='Quark',histtype='stepfilled')
    ax.set_title(title,color='white'); ax.set_xlabel(xl,color='#aaa')
    ax.legend(facecolor='#1a1a2e',labelcolor='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333355')
fig.suptitle('Point Cloud Statistics – Quark vs Gluon',color='white',fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/pc_stats.png',dpi=150,bbox_inches='tight',
                                 facecolor='#0d0d0d'); plt.show()

## Model: JetGNN (DGCNN / EdgeConv)

In [ ]:
class EdgeConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = EdgeConv(
            nn=nn.Sequential(
                nn.Linear(2*in_ch, out_ch, bias=False), nn.BatchNorm1d(out_ch), nn.LeakyReLU(0.2),
                nn.Linear(out_ch,  out_ch, bias=False), nn.BatchNorm1d(out_ch), nn.LeakyReLU(0.2),
            ), aggr='max')
        self.skip = nn.Sequential(nn.Linear(in_ch,out_ch,bias=False),
                                   nn.BatchNorm1d(out_ch)) if in_ch!=out_ch else nn.Identity()
        self.act = nn.LeakyReLU(0.2)
    def forward(self, x, ei): return self.act(self.conv(x,ei)+self.skip(x))


class JetGNN(nn.Module):
    """
    DGCNN-style classifier.
    Input node features (7): [η, φ, E_ECAL, E_HCAL, E_Tracks, E_tot, ΔR]
    """
    def __init__(self, in_ch=7, dropout=0.3):
        super().__init__()
        self.ec1 = EdgeConvBlock(in_ch, 64)
        self.ec2 = EdgeConvBlock(64, 128)
        self.ec3 = EdgeConvBlock(128, 256)
        self.clf = nn.Sequential(
            nn.Linear(512,256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(128,2))

    def forward(self, data):
        x,ei,b = data.x, data.edge_index, data.batch
        x = self.ec1(x,ei); x = self.ec2(x,ei); x = self.ec3(x,ei)
        x = torch.cat([global_mean_pool(x,b), global_max_pool(x,b)], dim=1)
        return self.clf(x)

model = JetGNN(in_ch=7, dropout=DROPOUT).to(DEVICE)
print(f'JetGNN params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Training

In [ ]:
opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS, eta_min=1e-5)
history = {k:[] for k in ['tl','ta','vl','va','vauc']}
best_auc = 0

def run_epoch(loader, train=True):
    model.train(train)
    tl=tc=tn=0; probs=[]; labs=[]
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False):
            batch=batch.to(DEVICE)
            if train: opt.zero_grad()
            logits=model(batch)
            loss=F.cross_entropy(logits,batch.y.squeeze())
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
            p=F.softmax(logits,1)[:,1]
            tl+=loss.item()*batch.num_graphs
            tc+=(logits.argmax(1)==batch.y.squeeze()).sum().item()
            tn+=batch.num_graphs
            probs.append(p.detach().cpu().numpy())
            labs.append(batch.y.squeeze().cpu().numpy())
    probs=np.concatenate(probs); labs=np.concatenate(labs)
    return tl/tn, tc/tn, roc_auc_score(labs,probs), probs, labs

for epoch in range(1, N_EPOCHS+1):
    tl,ta,_,_,_ = run_epoch(train_loader, train=True)
    vl,va,vauc,_,_ = run_epoch(val_loader, train=False)
    sched.step()
    for k,v in zip(['tl','ta','vl','va','vauc'],[tl,ta,vl,va,vauc]):
        history[k].append(v)
    if vauc > best_auc:
        best_auc=vauc
        torch.save(model.state_dict(), f'{OUT_DIR}/gnn_best.pt')
    if epoch%5==0 or epoch==1:
        print(f'Ep {epoch:3d} | train loss {tl:.4f} acc {ta:.4f} | '
              f'val loss {vl:.4f} acc {va:.4f} AUC {vauc:.4f}')
model.load_state_dict(torch.load(f'{OUT_DIR}/gnn_best.pt',map_location=DEVICE))
print(f'\nBest val AUC: {best_auc:.4f}')

## Training Curves

In [ ]:
epochs=range(1,len(history['tl'])+1)
fig,axes=plt.subplots(1,3,figsize=(14,4),facecolor='#0d0d0d')
specs=[('Loss','tl','vl'),('Accuracy','ta','va'),('Val AUC','vauc',None)]
for ax,(ttl,tk,vk) in zip(axes,specs):
    ax.plot(epochs,history[tk],color='#00e5ff',lw=1.8,label='Train')
    if vk: ax.plot(epochs,history[vk],color='#ff6e40',lw=1.8,ls='--',label='Val')
    ax.set_title(ttl,color='white'); ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white'); ax.set_xlabel('Epoch',color='#aaa')
    if vk: ax.legend(facecolor='#1a1a2e',labelcolor='white')
fig.suptitle('JetGNN Training Curves',color='white',fontsize=13)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png',dpi=150,
    bbox_inches='tight',facecolor='#0d0d0d'); plt.show()

## Test Set Evaluation

In [ ]:
te_loss,te_acc,te_auc,te_probs,te_labels = run_epoch(test_loader,train=False)
print(f'Test Loss: {te_loss:.4f}  Acc: {te_acc:.4f} ({te_acc*100:.2f}%)  AUC: {te_auc:.4f}')
print(classification_report(te_labels,(te_probs>=0.5).astype(int),
                            target_names=['Gluon','Quark']))

## ROC Curve

In [ ]:
fpr,tpr,_ = roc_curve(te_labels,te_probs)
fig,ax=plt.subplots(figsize=(6,6),facecolor='#0d0d0d'); ax.set_facecolor('#1a1a2e')
ax.plot(fpr,tpr,color='#00e5ff',lw=2,label=f'JetGNN  AUC={te_auc:.4f}')
ax.plot([0,1],[0,1],color='#555',ls='--')
ax.set_xlabel('FPR',color='#aaa'); ax.set_ylabel('TPR',color='#aaa')
ax.set_title('ROC – Quark/Gluon',color='white'); ax.tick_params(colors='white')
ax.legend(facecolor='#1a1a2e',labelcolor='white')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/roc.png',dpi=150,bbox_inches='tight',
    facecolor='#0d0d0d'); plt.show()

## Score Distribution & Confusion Matrix

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5),facecolor='#0d0d0d')

# Score distribution
ax=axes[0]; ax.set_facecolor('#1a1a2e'); ax.tick_params(colors='white')
bins=np.linspace(0,1,50)
ax.hist(te_probs[te_labels==0],bins=bins,density=True,alpha=0.6,
        color='#ff6e40',label='Gluon',histtype='stepfilled')
ax.hist(te_probs[te_labels==1],bins=bins,density=True,alpha=0.6,
        color='#00e5ff',label='Quark',histtype='stepfilled')
ax.set_xlabel('P(Quark)',color='#aaa'); ax.set_title('Score Distribution',color='white')
ax.legend(facecolor='#1a1a2e',labelcolor='white')

# Confusion matrix
ax2=axes[1]; ax2.set_facecolor('#1a1a2e')
cm=confusion_matrix(te_labels,(te_probs>=0.5).astype(int))
im=ax2.imshow(cm,cmap='Blues'); plt.colorbar(im,ax=ax2)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Gluon','Quark'],color='white')
ax2.set_yticklabels(['Gluon','Quark'],color='white')
ax2.set_xlabel('Predicted',color='#aaa'); ax2.set_ylabel('True',color='#aaa')
ax2.set_title('Confusion Matrix',color='white')
for i in range(2):
    for j in range(2):
        ax2.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',fontsize=13,
                 color='white' if cm[i,j]<cm.max()*0.6 else 'black')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/eval.png',dpi=150,bbox_inches='tight',
    facecolor='#0d0d0d'); plt.show()